In [39]:
import numpy as np
import pandas as pd

from SymbolicDSGE import DSGESolver, ModelParser, Shock, SolvedModel
from SymbolicDSGE.monte_carlo import (
    MCPipeline,
    MCData,
    MCContext,
    numpy_operation,
    pandas_operation,
)
from SymbolicDSGE.monte_carlo.operations.core import (
    reference_filter_step,
    simulation_step,
)
from SymbolicDSGE.monte_carlo.operations.regressions import regression_step
from SymbolicDSGE.monte_carlo.operations.tests import (
    ljung_box_test_step,
    wald_test_step,
)
from SymbolicDSGE.monte_carlo.operations.postproc import postproc_step
from SymbolicDSGE.monte_carlo.operations.transforms import transform_step
import cProfile

In [40]:
model, kalman = ModelParser("../../MODELS/misspec_test/reference.yaml").get_all()

solver = DSGESolver(model, kalman)
compiled = solver.compile(
    n_state=3,
    n_exog=3,
)

steady_state = np.zeros(5, dtype=np.float64)
reference = solver.solve(compiled, steady_state=steady_state)

dgp_model, dgp_kalman = ModelParser("../../MODELS/misspec_test/misspec.yaml").get_all()
dgp_comp = DSGESolver(dgp_model, dgp_kalman).compile(n_state=3, n_exog=3)
dgp_sol = DSGESolver(dgp_model, dgp_kalman)
dgp = dgp_sol.solve(dgp_comp, steady_state=steady_state)

In [41]:
T = 200
n_obs = len(reference.compiled.observable_names)


@numpy_operation
def custom_standardize(
    *, context: MCContext, reference: SolvedModel, dgp: SolvedModel, rep_idx: int
):
    del reference, dgp, rep_idx
    data: MCData = context.require_data()
    obs = data.observables
    if obs is not None:
        return (obs - obs.mean(axis=0)) / obs.std(axis=0)
    return None


@pandas_operation
def get_std_obs_mean(*, traces):
    return traces["payload.standardize_observables"].mean(axis=0)


pipeline = MCPipeline(
    per_rep_steps=[
        simulation_step(
            T=T,
            shocks={
                "g,z": Shock(T=T, dist="norm", multivar=True, seed=0),
                "r": Shock(T=T, dist="norm", seed=1),
            },
            observables=True,
        ),
        reference_filter_step(estimate_R_diag=False),
        transform_step(
            "standardize_observables",
            func=custom_standardize,
        ),
        wald_test_step(
            "std_innov_mean",
            source="std_innov",
            target=np.zeros(n_obs),
            kind="mean",
            burn_in=20,
        ),
        wald_test_step(
            "std_innov_second_moment",
            source="std_innov",
            target=np.eye(n_obs),
            kind="second_moment",
            burn_in=20,
        ),
        ljung_box_test_step(
            "OGap_autocorr",
            source="std_innov",
            column=0,
            lags=10,
            burn_in=20,
        ),
        regression_step(
            "OutGap_regression",
            kind="ridge_gs",
            start=1e-6,
            stop=1e2,
            num=10,
            X_source="x_pred",
            y_source="innov",
            y_column=0,
            X_columns=[2, 3, 4],
            variables=["r", "x", "Pi"],
        ),
    ],
    postproc_steps=[
        postproc_step(
            "std_obs_mean",
            func=get_std_obs_mean,
        )
    ],
)

In [42]:
mc = pipeline.run(
    reference=reference,
    dgp=dgp,
    n_rep=1000,
    retain_payloads=False,
    retain_test_results=False,
    verbosity=2,
)

datagen concluded successfully with 404.53 it/s.
filter concluded successfully with 469.04 it/s.
standardize_observables concluded successfully with 14257.89 it/s.
std_innov_mean concluded successfully with 18557.70 it/s.
std_innov_second_moment concluded successfully with 16384.79 it/s.
OGap_autocorr concluded successfully with 28913.18 it/s.
OutGap_regression concluded successfully with 9146.72 it/s.
std_obs_mean concluded successfully with 3055.30 it/s.


In [43]:
t_summary = pd.DataFrame(
    {
        name: {
            "mean_statistic": res.mean_statistic,
            "mean_pval": res.mean_pval,
            "rejection_rate": res.rejection_rate,
            "ci_low": res.pval_confidence_interval()[0],
            "ci_high": res.pval_confidence_interval()[1],
        }
        for name, res in mc.test_summaries.items()
    }
).T
t_summary.round(3)

,mean_statistic,mean_pval,rejection_rate,ci_low,ci_high
std_innov_mean,2.698,0.555,0.053,0.041,0.069
std_innov_second_moment,720.891,0.000,1.000,0.996,1.000
OGap_autocorr,25.197,0.035,0.805,0.779,0.828


In [44]:
print(
    any(
        res.value != 0
        for res in mc.regression_summaries["OutGap_regression"].status_trace
    )
)

(
    mc.regression_summaries["OutGap_regression"].coefficients.mean(axis=0).round(2),
    mc.regression_summaries["OutGap_regression"].r2_trace.mean().round(4),
)

False


(array([-0.  , -0.32, -0.18, -2.8 ]), np.float64(0.1266))